# ABUK Neural Network Trading Strategy - Enhanced

This notebook investigates whether a neural network can learn patterns
from the historical data of ABUK stock and use those predictions
to generate a simple trading strategy.

Compared to the baseline version, this enhanced pipeline adds:

1. **Rich feature set** - returns, price/SMA ratios, RSI, volatility,
   MACD histogram, multi-day momentum, and volume ratio.
2. **5-day forward return target** - predicting a 5-day horizon instead
   of 1-day smooths out daily noise and gives the model more signal.
3. **Feature standardization** - z-score normalization fit on the training
   set only, so the model sees well-scaled inputs.
4. **Train / validation / test split** - a validation set is used for early
   stopping and model selection; the test set is opened only once at the end.
5. **Regularized architecture** - BatchNorm + Dropout to fight overfitting.
6. **Mini-batch training** - with weight decay and a learning-rate scheduler
   for more stable convergence.
7. **Ensemble prediction** - average predictions from 3 models trained with
   different seeds to reduce variance.
8. **Better evaluation** - directional accuracy, information coefficient,
   Sharpe ratio, and a threshold-based trading rule.

The overall pipeline is:

Historical Features -> Neural Network -> Predicted 5-day Return
-> Trading Signal -> Portfolio Performance

## 1. Import Libraries and Load Data

We use the same `DataFeed` from the previous trading notebooks so that
the neural-network experiment works with the same market data.

For this experiment, we focus only on ABUK, which is asset index 0.

In [ ]:
import sys, os

while not os.path.isdir("src") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed
from tradinglab.indicators import ema, sma, rsi, rolling_volatility
from tradinglab.metrics import (
    total_return,
    annualized_return,
    volatility,
    sharpe,
    max_drawdown,
    directional_accuracy,
    information_coefficient,
)

torch.manual_seed(0)
np.random.seed(0)

## 2. Load Data and Build Features

We load ABUK and build a rich feature matrix. Each row `t` holds the
features known at the close of day `t`, and the label is the **5-day
forward return** from day `t` to day `t+5`.

The feature set (in order):

    0: return        daily simple return
    1: p/sma_fast    (close / SMA_10) - 1
    2: p/sma_slow    (close / SMA_30) - 1
    3: rsi           RSI(14) / 100
    4: volatility    rolling 20-day std of returns
    5: macd_hist     MACD(12,26) line minus its 9-day signal
    6: return_5d     5-day cumulative return
    7: return_10d    10-day cumulative return
    8: volume_ratio  today's volume / 20-day average volume

In [ ]:
feed = DataFeed.from_dir(
    "data/egx",
    symbols=["ABUK"]
)

print("symbols:", feed.symbols)
print("number of assets:", feed.n_assets)
print("number of days:", feed.n_days)

In [ ]:
# ---- build the feature matrix for ABUK (asset index 0) ----
close = feed.close[:, 0]
ret = feed.returns[:, 0]
vol = feed.volume[:, 0]
n = feed.n_days

# MACD histogram
ema12 = ema(close, 12)
ema26 = ema(close, 26)
macd_line = ema12 - ema26
first_valid = int(np.argmax(~np.isnan(macd_line)))
macd_signal = np.full(n, np.nan)
macd_signal[first_valid:] = ema(macd_line[first_valid:], 9)
macd_hist = macd_line - macd_signal

# Multi-day momentum
ret5 = np.full(n, np.nan)
ret5[5:] = close[5:] / close[:-5] - 1.0
ret10 = np.full(n, np.nan)
ret10[10:] = close[10:] / close[:-10] - 1.0

# Volume ratio: today's volume / 20-day average
vol_avg20 = np.full(n, np.nan)
for i in range(19, n):
    vol_avg20[i] = vol[i-19:i+1].mean()
volume_ratio = vol / vol_avg20

FEATURE_NAMES = ["return", "p/sma_fast", "p/sma_slow", "rsi", "volatility",
                 "macd_hist", "return_5d", "return_10d", "volume_ratio"]

X_full = np.column_stack([
    ret,
    close / sma(close, 10) - 1.0,
    close / sma(close, 30) - 1.0,
    rsi(close, 14) / 100.0,
    rolling_volatility(ret, 20),
    macd_hist,
    ret5,
    ret10,
    volume_ratio,
])
n_features = X_full.shape[1]

# Label: 5-day forward return (last 5 days have no label -> NaN)
HORIZON = 5
y_full = np.full(n, np.nan)
y_full[:-HORIZON] = close[HORIZON:] / close[:-HORIZON] - 1.0

print("X_full shape:", X_full.shape)
print("y_full shape:", y_full.shape)
print("feature names:", FEATURE_NAMES)
print("\nFirst 3 feature rows (may contain NaN during warm-up):")
print(pd.DataFrame(X_full[:3], columns=FEATURE_NAMES))

## 3. Drop NaN Rows

The first ~30 rows are NaN because indicators like SMA(30) and MACD need
warm-up history. We drop any row with a missing feature or label.

In [ ]:
valid = ~np.isnan(X_full).any(axis=1) & ~np.isnan(y_full)

X = X_full[valid].astype(np.float32)
y = y_full[valid].astype(np.float32)
days = np.arange(feed.n_days)[valid]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("first valid day index:", days[0])

## 4. Chronological Train / Validation / Test Split

Because stock prices are time-series data, we must preserve the
chronological order. We split into three contiguous periods:

    Past                              Future
    ----------------------------------|----------|----------
              Training              | Validation|  Testing
                 60%                |    20%    |    20%

The **validation** set is used for early stopping and hyperparameter
selection. The **test** set is opened only once, at the very end, to
report the honest result.

In [ ]:
n = len(X)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

Xtr, ytr = X[:train_end], y[:train_end]
Xva, yva = X[train_end:val_end], y[train_end:val_end]
Xte, yte = X[val_end:], y[val_end:]

print(f"training samples   : {len(Xtr)}")
print(f"validation samples : {len(Xva)}")
print(f"testing samples    : {len(Xte)}")

## 5. Standardize Features

Neural networks converge much faster when inputs are roughly zero-mean,
unit-variance. We compute the mean and std **on the training set only**
and apply the same transform to validation and test - this prevents
information leakage from the future.

In [ ]:
mu = Xtr.mean(axis=0)
sigma = Xtr.std(axis=0) + 1e-8

Xtr = (Xtr - mu) / sigma
Xva = (Xva - mu) / sigma
Xte = (Xte - mu) / sigma

print("feature means after standardization (train):")
print(np.round(Xtr.mean(axis=0), 4))
print("\nfeature stds after standardization (train):")
print(np.round(Xtr.std(axis=0), 4))

## 6. Define the Model

We use a compact MLP with **BatchNorm** (stabilizes training) and
**Dropout** (regularizes against overfitting). The architecture:

    Input (n_features)
      -> Linear(32) -> BatchNorm -> ReLU -> Dropout(0.2)
      -> Linear(16) -> BatchNorm -> ReLU -> Dropout(0.2)
      -> Linear(1)

A compact model is deliberate: with ~1700 training samples, a smaller
network generalizes better than a large one.

Dropout is active only during training (`model.train()`), not during
evaluation (`model.eval()`).

In [ ]:
class MLP(nn.Module):

    def __init__(self, input_size, dropout=0.2):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

## 7. Train an Ensemble with Mini-Batches, Weight Decay, and Early Stopping

We train **3 models with different random seeds** and average their
predictions. Ensembling reduces prediction variance - a cheap and
effective way to improve robustness.

Each model is trained with:

- **Adam** optimizer with **weight decay** (L2 regularization)
- **Mini-batches** of 64 for more stable gradients
- **ReduceLROnPlateau** scheduler that lowers the learning rate when
  validation loss stalls
- **Early stopping**: if validation loss doesn't improve for 30 epochs,
  we stop and restore the best weights

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
EPOCHS = 500
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 30
N_SEEDS = 3

Xtr_t = torch.tensor(Xtr)
ytr_t = torch.tensor(ytr)
Xva_t = torch.tensor(Xva)
yva_t = torch.tensor(yva)
Xte_t = torch.tensor(Xte)
yte_t = torch.tensor(yte)

loss_fn = nn.MSELoss()

def train_one(seed):
    """Train a single model with a given seed; return (model, train_hist, val_hist)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLP(n_features)

    train_ds = TensorDataset(Xtr_t, ytr_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=10,
    )

    train_hist = []
    val_hist = []
    best_val = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(EPOCHS):
        # ---- training pass over mini-batches ----
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        train_loss = epoch_loss / n_batches
        train_hist.append(train_loss)

        # ---- validation pass ----
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xva_t), yva_t).item()
        val_hist.append(val_loss)

        scheduler.step(val_loss)

        # ---- early stopping ----
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, train_hist, val_hist, best_val

models = []
all_train_hist = []
all_val_hist = []

for seed in range(N_SEEDS):
    m, th, vh, bv = train_one(seed)
    models.append(m)
    all_train_hist.append(th)
    all_val_hist.append(vh)
    print(f"seed {seed}: best val loss = {bv:.6f}, epochs = {len(th)}")

In [ ]:
# Average the loss curves across the ensemble
min_len = min(len(h) for h in all_train_hist)
train_hist = np.mean([h[:min_len] for h in all_train_hist], axis=0)
val_hist = np.mean([h[:min_len] for h in all_val_hist], axis=0)

plt.figure(figsize=(10, 4))

plt.plot(train_hist, label="train loss (avg)")
plt.plot(val_hist, label="validation loss (avg)")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("ABUK Neural Network - Training vs Validation Loss (Ensemble Avg)")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

## 8. Evaluate on the Test Set

Now we open the test set - the period the models never saw during
training or validation. We average the predictions from all ensemble
members and report:

- **MSE** - how close the predicted 5-day return is to the actual
- **Directional accuracy** - fraction of days the sign (up/down) is right
- **Information coefficient (IC)** - correlation between predicted and
  actual returns

In [ ]:
with torch.no_grad():
    preds = [m(Xte_t).numpy() for m in models]
    predictions = np.mean(preds, axis=0)
    test_mse = loss_fn(torch.tensor(predictions), yte_t).item()

actual = yte

print(f"test MSE              : {test_mse:.6f}")
print(f"directional accuracy  : {directional_accuracy(predictions, actual):.2%}")
print(f"information coeff.    : {information_coefficient(predictions, actual):.4f}")

In [ ]:
SHOW_N = 150

plt.figure(figsize=(12, 4))

plt.plot(
    actual[:SHOW_N],
    label="actual 5-day forward return"
)

plt.plot(
    predictions[:SHOW_N],
    label="NN ensemble prediction",
    linestyle="--"
)

plt.axhline(0, linewidth=0.8)

plt.title("ABUK - Neural Network Prediction vs Actual (Test Set)")
plt.xlabel("Test day")
plt.ylabel("Return")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

## 9. Trading Strategy

We use a **confidence threshold**: we only go long when the model's
predicted 5-day return is strong enough to be worth acting on.
Otherwise we stay in cash. This avoids trading on noise.

    weights = 1.0  if  prediction >  threshold
              0.0  otherwise (stay in cash)

We compare against a **buy & hold** benchmark.

Note: the 5-day strategy rebalances every day based on the next 5-day
view, but we apply the position each day to keep things simple.

In [ ]:
THRESHOLD = 0.0  # go long when the model predicts positive 5-day return

weights = np.where(predictions > THRESHOLD, 1.0, 0.0)

print(f"fraction of days in market: {weights.mean():.2%}")

In [ ]:
strategy_returns = weights * actual

strategy_curve = np.cumprod(1 + strategy_returns)
buy_hold_curve = np.cumprod(1 + actual)

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    strategy_curve,
    label="NN strategy"
)

plt.plot(
    buy_hold_curve,
    label="ABUK buy & hold",
    linestyle="--"
)

plt.title("ABUK - Neural Network Strategy vs Buy & Hold (Test Set)")
plt.xlabel("Test day")
plt.ylabel("Portfolio value (normalized)")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

## 10. Performance Metrics

We compare the NN strategy against buy & hold using the standard
metrics from `metrics.py`.

In [ ]:
print("=== NN Strategy ===")
print(f"total return     : {total_return(strategy_returns):+.2%}")
print(f"annualized return: {annualized_return(strategy_returns):+.2%}")
print(f"volatility       : {volatility(strategy_returns):.2%}")
print(f"sharpe ratio     : {sharpe(strategy_returns):.2f}")
print(f"max drawdown     : {max_drawdown(strategy_returns):.2%}")

print("\n=== Buy & Hold ===")
print(f"total return     : {total_return(actual):+.2%}")
print(f"annualized return: {annualized_return(actual):+.2%}")
print(f"volatility       : {volatility(actual):.2%}")
print(f"sharpe ratio     : {sharpe(actual):.2f}")
print(f"max drawdown     : {max_drawdown(actual):.2%}")

## 11. Summary

The enhanced pipeline demonstrates the full workflow:

1. **Features matter** - using a rich technical-indicator feature set
   (returns, SMA ratios, RSI, volatility, MACD, momentum, volume) gives
   the model much more signal than raw returns alone.
2. **Horizon selection** - predicting a 5-day forward return instead of
   1-day smooths daily noise and captures a more learnable signal.
3. **Standardization** - z-scoring features on the training set only
   improves convergence and prevents leakage.
4. **Validation + early stopping** - prevents overfitting and gives an
   honest way to pick the best model.
5. **Regularization** - BatchNorm and Dropout keep the model from
   memorizing noise.
6. **Ensembling** - averaging several models reduces variance and
   improves robustness.

The key takeaway: a good result comes from the whole pipeline - data,
features, training discipline, and evaluation - not just a bigger model.